# Distinguish readiness from liveness during a failure

Prerequisites: Pods, service endpoints, health checks and application dependencies.

A startup probe can give a slow application time to initialize before liveness and readiness checks take effect. Once started, readiness answers whether the container should receive normal service traffic; liveness answers whether the container is considered unhealthy enough for a restart under its configured policy. A pod can be running and live while unready.

The companion models one decision instant with three booleans. Started=true, ready=false, live=true removes normal endpoint eligibility without requesting a restart. Started=true, live=false requests a restart in this simplified model. Started=false suppresses the other decisions. The fixture omits failure thresholds, periods, initial delays, backoff, controller latency and service options, so it is not a Kubernetes emulator.

For a dependency outage, an indiscriminate liveness failure can cause restart loops without fixing the dependency. Investigate events, application responses and endpoint membership before changing probes. The downloadable manifest is an inspection fixture with no credentials; applying it to a real cluster requires a separate environment and image choice. No cluster deployment or recovery test is claimed by this notebook.

## Model structure

| Element | Model | Interpretation |
| --- | --- | --- |
| Startup | Initialization complete? | Gates subsequent checks |
| Readiness | Ready to receive traffic? | Endpoint eligibility |
| Liveness | Restart warranted? | Container recovery action |

## Predict

The default case is started but unready and live: eligible_endpoint=false, restart_requested=false. Flip only live to false and observe the independent restart decision.

In [ ]:
from pathlib import Path
import sys, math
base=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(base))
import systems_models as m

In [ ]:
result=m.kubernetes()
assert not result['eligible_endpoint']
assert not result['restart_requested']
assert m.kubernetes(live=False)['restart_requested']
assert not m.kubernetes(started=False,live=False)['restart_requested']
print(result)

## Perturb and explain

Write a dependency-outage timeline. Predict pod phase, readiness, service endpoints and restart count separately, then identify the read-only observations needed to test each prediction.

Record the changed input, prediction, result and a limit of your conclusion. Restart the kernel to reset the experiment.

## Check your reasoning

A live but unready container necessarily needs:

1. An immediate restart
2. Removal from normal ready endpoints, under the stated service model
3. Deletion of its persistent volume

Answer and explanations: 1: Readiness alone does not request a restart. 2: Traffic eligibility and process health are distinct. 3: Readiness does not authorize destructive storage changes.

## Primary evidence

- [Kubernetes — Configure Liveness, Readiness and Startup Probes](https://kubernetes.io/docs/tasks/configure-pod-container/configure-liveness-readiness-startup-probes/)

Review scope: this authored worked example and its stated assumptions; no hardware or production acceptance is implied.